In [1]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [2]:
!pip install -q langchain langchain-core langchain-community pydantic duckduckgo-search ddgs langchain-experimental requests langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 77.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
from langchain_core.tools import tool
import requests

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

/tmp/ipykernel_3256/914947371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [5]:
@tool
def get_weather_data(city:str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=0ff47689039b9b22cad26f73e19f4eec&query={city}'

  response=requests.get(url)

  return response.json()

In [6]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm)

In [7]:
from langchain_classic import hub
from langchain_classic.agents import create_react_agent, AgentExecutor

In [8]:
# step-1: Pull the ReAct prompt from the Langchain Hub

from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

In [9]:
# step-2: Create the ReAct agent manually with the pulled prompt
agent=create_react_agent(
    llm=llm,
    tools=[search_tool,get_weather_data],
    prompt=prompt
)

In [10]:
# step-3: Wrap it with AgentExecuter
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    verbose=True
)

In [11]:
# invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})



> Entering new AgentExecutor chain...
I need to first find the capital of Madhya Pradesh.
Action: duckduckgo_search
Action Input: capital of Madhya Pradesh
ObservBhopal: Madhya Pradesh will celebrate its Foundation Day on November 1 with a series of special events across the state, including the capital Bhopal. Chief Minister Dr. Mohan Yadav announced during a cabinet meeting on Tuesday that the celebrations will highlight the state's achievements over the past two years through a large-scale development exhibition and cultural performances. Bhopal is the capital city of Madhya Pradesh state, central India, Situated in the fertile plain of the Malwa Plateau, the city lies just north of the Vindhya Range. Madhya Pradesh, state of India that is situated in the heart of the country. It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive plateaus, and river valleys. The capital is Bhopal, in the west-central part of the state. The cap

In [12]:
response['output']

'The current weather in Bhopal, Madhya Pradesh, India is thundery with a temperature of 32°C, humidity at 46%, and wind blowing from the east at 21 km/h. There is no precipitation at the moment. The air quality is poor with a PM2.5 level of 121.5 and a PM10 level of 214.2. The UV index is 0, indicating low UV radiation.'